# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Building the feature vector the same way `scripts/01_prepare_features.py` does, but inline here so
the leakage test in section 3 is self-contained and every step is visible in one place.


In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd, numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

fv = df.copy()
fv = fv[(fv["impressions_90d"] > 0) & (fv["content_age_days"] >= 90)].drop_duplicates("content_id")

# engineered log features (traffic totals are heavy-tailed)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    fv[f"log_{col}"] = np.log1p(fv[col].fillna(0))

# has_-flags BEFORE any blind fillna, so the systematic missingness (Week-3 data contract, section 3)
# doesn't get silently absorbed into a plain 0
fv["has_clicks"] = (fv["clicks_90d"].fillna(0) > 0).astype(int)
fv["has_ai_sessions"] = (fv["ai_sessions_90d"].fillna(0) > 0).astype(int)
fv["measurable_opportunity"] = ((fv["impressions_90d"] >= 100) & (fv["sessions_90d"].fillna(0) > 0)).astype(int)

fv["is_declining_label"] = fv["trend_direction"].str.lower().eq("down").astype(int)

for col in MODEL_NUMERIC_FEATURES:
    fv[col] = pd.to_numeric(fv[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for col in MODEL_CATEGORICAL_FEATURES:
    fv[col] = fv[col].fillna("unknown").astype(str)

print(fv.shape[0], "pages after the same filters the reference pipeline uses")
print("Declining rate:", round(fv["is_declining_label"].mean(), 3))


30000 pages after the same filters the reference pipeline uses
Declining rate: 0.542


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before the decision? |
|---|---|---|---|
| `log_impressions_90d` / `log_clicks_90d` / `log_sessions_90d` / `log_ai_sessions_90d` | log1p of 90d totals | counts default to 0, no true missing | Yes -- observed activity, not a future outcome |
| `days_with_impressions` / `days_with_sessions` | how many of the 90 days had any activity | no missing | Yes |
| `content_age_days` / `days_since_last_update` | page lifecycle | no missing in this slice | Yes -- structural, always known |
| `ctr`, `avg_position` | search performance | `avg_position=0` means "no data"; kept as 0 which must be read as "unknown," never "great position" | Yes, but `avg_position==0` needs a guard before use |
| `engagement_rate`, `scroll_rate` | GA4 engagement | blank when `sessions_90d`/`pageviews_90d` = 0 -> filled 0 | Yes |
| `ai_traffic_pct` | AI-referred session share | blank when `sessions_90d`=0 -> filled 0 | Yes, but sparse (see `docs/data-dictionary.md`) |
| `search_volume`, `competition`, `cpc` | keyword context | blank for rows with no keyword data, and that blank pattern follows `content_type` (checked in the Week-3 data contract), not chance | Yes -- keyword metadata predates any performance |
| `content_type`, `main_intent`, `competition_level`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier` | categorical context / buckets | filled `"unknown"` | Yes |

`has_clicks`, `has_ai_sessions`, `measurable_opportunity` are engineered 0/1 flags built specifically
so the systematic missingness stays visible instead of a blind `fillna(0)` erasing it.


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

X_clean = pd.get_dummies(fv[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES], columns=MODEL_CATEGORICAL_FEATURES)
y = fv["is_declining_label"]

clean_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_clean, y)
clean_auc = roc_auc_score(y, clean_tree.predict_proba(X_clean)[:, 1])
print(f"Clean feature set (no label-adjacent columns) -- in-sample ROC-AUC: {clean_auc:.3f}")
print("A realistic, not-suspiciously-perfect score is the first leakage smell test -- passes.")


Clean feature set (no label-adjacent columns) -- in-sample ROC-AUC: 0.709
A realistic, not-suspiciously-perfect score is the first leakage smell test -- passes.


## 3. The leakage hunt

Attacking my own feature set: what happens if I "accidentally" add back the column the label is
computed from (`trend_pct`) and its immediate sub-window siblings (`impressions_last_30d`,
`impressions_prev_30d`)?


In [3]:
leaky_cols = ["trend_pct", "impressions_last_30d", "impressions_prev_30d"]
X_leaky = pd.get_dummies(
    fv[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES + leaky_cols],
    columns=MODEL_CATEGORICAL_FEATURES,
)
leaky_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_tree.predict_proba(X_leaky)[:, 1])

print(f"Clean ROC-AUC: {clean_auc:.3f}")
print(f"Leaky ROC-AUC: {leaky_auc:.3f}  <- jumps toward 1.0")

importances = pd.Series(leaky_tree.feature_importances_, index=X_leaky.columns).sort_values(ascending=False)
print("\nTop leaky-model feature:", importances.index[0], f"({importances.iloc[0]:.3f} importance)")
print("-> trend_pct dominates and the score inflates -- exactly the leakage signature")
print("   'one feature towers over all others, and the score is near-perfect.'")
print("Confirms trend_pct / trend_direction / the last30-prev30 sub-windows stay OUT of the real feature set.")


Clean ROC-AUC: 0.709
Leaky ROC-AUC: 1.000  <- jumps toward 1.0

Top leaky-model feature: trend_pct (1.000 importance)
-> trend_pct dominates and the score inflates -- exactly the leakage signature
   'one feature towers over all others, and the score is near-perfect.'
Confirms trend_pct / trend_direction / the last30-prev30 sub-windows stay OUT of the real feature set.


## 4. What I excluded and why

- `trend_direction`, `trend_pct` -- the label and the number it's computed from. Never features.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`,
  `clicks_prev_30d`, `sessions_prev_30d` -- the exact sub-windows `trend_pct` is built from;
  confirmed above they inflate the score the same way `trend_pct` does.
- `content_id`, `client_id` -- pseudonymous IDs; grouping/join/split keys only, never model inputs.
- `provider_used`, `model_used` -- which LLM/provider wrote the article; an authoring detail, not a
  search signal, and the data dictionary marks both "not a model feature."
- Any FlyRank product decision flag (`health_score`, `priority_score`, `action_type`, etc.) -- not
  shipped in this dataset at all, so there's nothing to strip out here, but noted so a future,
  richer data source doesn't get one folded in by accident.


In [4]:
excluded = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_id", "client_id", "provider_used", "model_used",
]
final_features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
overlap = set(excluded) & set(final_features)
print("Excluded/feature overlap (must be empty):", overlap)
print(f"\nFinal feature count: {len(final_features)} numeric+categorical columns, "
      f"{len(excluded)} explicitly excluded with reasons above.")


Excluded/feature overlap (must be empty): set()

Final feature count: 26 numeric+categorical columns, 12 explicitly excluded with reasons above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.